# PIIS: Cryptographic Audit Ledger & RFC 8785 Verification

This notebook demonstrates the cryptographic integrity architecture of PIIS.

### Integrity Guarantees
- **RFC 8785 Canonicalization (JCS):** Guarantees byte-level JSON serialization determinism across different runtimes.
- **SHA-256 Hash Chaining:** Every entry links to the previous entry digest, preventing undetected record insertion or deletion.
- **HMAC-SHA256 Notarization:** Authenticated review receipts prove authorization by genuine human operators.
- **Monotonic Checkpoint Anchor:** Verifies that the ledger has not suffered truncation or state rollbacks.


In [ ]:
import sys
import os
import json

sys.path.insert(0, os.path.abspath('../src'))

from piis.audit.canonical import CanonicalJsonSerializer
from piis.investigation.approval import HumanApprovalWorkflow
from piis.investigation.schemas import DiagnosticRecommendation, RecommendationRisk

print('Audit and cryptographic modules imported.')


## 1. RFC 8785 Canonical JSON Serialization Proof
Demonstrate how key reordering or whitespace changes produce an identical cryptographic byte representation.


In [ ]:
obj1 = {'b_field': 100, 'a_field': 'telecom', 'nested': {'y': 2, 'x': 1}}
obj2 = {'nested': {'x': 1, 'y': 2}, 'a_field': 'telecom', 'b_field': 100}

serializer = CanonicalJsonSerializer()
canonical_bytes1 = serializer.serialize(obj1)
canonical_bytes2 = serializer.serialize(obj2)
digest1 = serializer.compute_digest(obj1)
digest2 = serializer.compute_digest(obj2)

print('Canonical Bytes Match:', canonical_bytes1 == canonical_bytes2)
print('SHA-256 Digest Match:', digest1 == digest2)
print('Canonical Output:', canonical_bytes1.decode('utf-8'))


## 2. Review Decision Signing and Ledger Chaining
Register a diagnostic recommendation and simulate an authenticated human operator approval.


In [ ]:
workflow = HumanApprovalWorkflow()
rec = DiagnosticRecommendation(
    recommendation_id='REC-AUDIT-001',
    description='Perform RF feeder realignment on Cell 1',
    reason='VSWR exceeds 1.8 baseline',
    supporting_evidence_ids=['EVID-001'],
    source_runbook_id='SOP-RAN-VSWR-001',
    risk_classification=RecommendationRisk.READ_ONLY_SAFE,
    human_approval_required=True
)
workflow.register_recommendation(rec, incident_id='INC-SNOW-INC0089201')

receipt = workflow.approve_recommendation(
    recommendation_id='REC-AUDIT-001',
    reviewer_id='noc-operator-01',
    review_notes='Verified RF line telemetry; feeder replacement approved.'
)

print('Approval Receipt Generated:')
print('  Action:', receipt.action)
print('  Sequence ID:', receipt.sequence_id)
print('  Record Digest (SHA-256):', receipt.record_digest)
print('  Previous Digest:', receipt.previous_record_digest)
print('  HMAC Signature:', receipt.hmac_signature[:32] + '...')
